# Transfer Learning — Oxford Flowers102 Clean Solution

Compares a CNN trained from scratch vs ResNet18 transfer learning (freeze backbone, train new head).

**Key concepts:** ImageNet normalization, freezing layers, replacing FC head, fine-tuning only the classifier.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 1: Dataset

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.Flowers102(root='data', split='train', download=True, transform=transform)
val_dataset = datasets.Flowers102(root='data', split='val', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## Step 2: CNN from Scratch

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 56 * 56, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_scratch = FlowerCNN().to(device)
loss_fn = nn.NLLLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=0.0001)

In [ ]:
def train_model(model, loader, loss_fn, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(loader):.4f}, Acc: {100*correct/total:.2f}%")

train_model(model_scratch, train_loader, loss_fn, optimizer, epochs=8)

## Step 3: Transfer Learning with ResNet18

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 102)
model = model.to(device)

loss_fn_tl = nn.CrossEntropyLoss()
optimizer_tl = optim.Adam(model.fc.parameters(), lr=0.001)  # only new head

In [ ]:
train_model(model, train_loader, loss_fn_tl, optimizer_tl, epochs=6)

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")

evaluate(model, val_loader)